In [1]:
import os
import matplotlib.pyplot as plt
from datetime import datetime
import geopandas as gpd
import xarray as xr
from blackmarble.extract import bm_extract
from blackmarble.raster import bm_raster
from datetime import datetime, timedelta
import random
from dateutil.relativedelta import relativedelta
from pathlib import Path
from dotenv import load_dotenv
import sys
sys.path.append('../src')
from ntl_functions import plot_NASA_NTL, filter_dataset_by_bounding_box, mask_dataset_by_geometry


load_dotenv("../.env")
bearer = os.getenv("NASA_API_BEARER_TOKEN")
if bearer is None:
    raise ValueError("NASA_API_BEARER_TOKEN not found. Please create and enter a bearer token into .env")

plt.rcParams["figure.figsize"] = (18, 10)
run_date = datetime.today().strftime("%Y%m%d")

In [2]:
os.getcwd()
base_directory = Path("C:\\Users\\samsa\\Documents")
# Define the specific output directory within the base directory
output_directory = base_directory / "black_marble_output_full"
# Ensure the output directory exists
output_directory.mkdir(parents=True, exist_ok=True)
print(f"Output Directory: {output_directory.resolve()}")

Output Directory: C:\Users\samsa\Documents\black_marble_output_full


Data Download

In [ ]:
#Somalia Shape File Downlaoded from.
# We used three different shapefiles during our analysis. 
# Comparisons between these can be found in the codebook titled: "Somaliland_Shapefile_Comparisons" 
# Use either somaliland_shp_berbera or gdf_expanded for the analysis.
#https://gadm.org/download_country.html
gdf = gpd.read_file(
    "zip://../data/Combined_Datasets/gadm41_SOM_1.json.zip"
)


#Somaliland SHP File
#somaliland_shp = gdf[gdf["HASC_1"].isin(["SO.AW", "SO.WO", "SO.TO", "SO.SA", "SO.SO"])]

#Phillip Somaliland_SHP
somaliland_shp_berbera = gpd.read_file(
    "../data/Combined_Datasets/Somaliland_with_berbera/somaliland_with_berbera.shp"
)

"""gdf_expanded = (
    somaliland_shp
    .to_crs(epsg=32638)
    .dissolve()
    .buffer(300)
    .to_crs(epsg=4326)
    .to_frame("geometry")
    .reset_index(drop=True)
)"""

# Annual Radiance Data VNP46A4

In [ ]:
# Annual data: raster for 2012-2024
SL_VNP46A4_EY_2012_24 = bm_raster(
    somaliland_shp_berbera, product_id="VNP46A4", date_range=["2012-01-01","2013-01-01", "2014-01-01", "2015-01-01",
                                           "2016-01-01","2017-01-01", "2018-01-01", "2019-01-01",
                                           "2020-01-01","2021-01-01", "2022-01-01", "2023-01-01", "2024-01-01"], token=bearer,
)

#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
SL_VNP46A4_EY_2012_24.to_netcdf(f"../data/NTL_Data/{run_date}_SL_Berbera_VNP46A4_EY_2012_24.nc")

'# Annual data: raster for 2012-2024\nSL_VNP46A4_EY_2012_24 = bm_raster(\n    somaliland_shp_berbera, product_id="VNP46A4", date_range=["2012-01-01","2013-01-01", "2014-01-01", "2015-01-01",\n                                           "2016-01-01","2017-01-01", "2018-01-01", "2019-01-01",\n                                           "2020-01-01","2021-01-01", "2022-01-01", "2023-01-01", "2024-01-01"], token=bearer,\n)\n\n#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. \nSL_VNP46A4_EY_2012_24.to_netcdf(f"../data/NTL_Data_A4/{run_date}_SL_Berbera_VNP46A4_EY_2012_24.nc")'

# Downloading Annual Num Obvs Data

In [ ]:
# Data Number of Obvs Download
#Code to extract observation data

yearly_obvs = bm_raster(
    somaliland_shp_berbera, 
    product_id="VNP46A4", 
    date_range=["2012-01-01","2013-01-01", 
                "2014-01-01", "2015-01-01",
                "2016-01-01","2017-01-01", "2018-01-01", "2019-01-01",
                "2020-01-01","2021-01-01", "2022-01-01", 
                "2023-01-01", "2024-01-01",], 
    token=bearer,
    file_directory = output_directory,
    variable="NearNadir_Composite_Snow_Free_Num",
)

#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
yearly_obvs.to_netcdf(f"../data/NTL_Data/{run_date}_Num_Obvs_SL_Berbera_VNP46A4_EY_2012_24.nc")

# Downloading Annual Quality Data

In [ ]:
# Data Quality Download
#Code to extract quality data

quality_r = bm_raster(
    somaliland_shp_berbera,
    product_id="VNP46A4",
    date_range=["2012-01-01","2013-01-01", 
                "2014-01-01", "2015-01-01",
                "2016-01-01","2017-01-01", "2018-01-01", "2019-01-01",
                "2020-01-01","2021-01-01", "2022-01-01", 
                "2023-01-01", "2024-01-01",], 
    token=bearer,
    variable="NearNadir_Composite_Snow_Free_Quality",
)

#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
quality_r.to_netcdf(f"../data/NTL_Data/{run_date}_Quality_SL_Berbera_VNP46A4_2012_24.nc")

'\nquality_r = bm_raster(\n    somaliland_shp_berbera,\n    product_id="VNP46A4",\n    date_range=["2012-01-01","2013-01-01", \n                "2014-01-01", "2015-01-01",\n                "2016-01-01","2017-01-01", "2018-01-01", "2019-01-01",\n                "2020-01-01","2021-01-01", "2022-01-01", \n                "2023-01-01", "2024-01-01",], \n    token=bearer,\n    variable="NearNadir_Composite_Snow_Free_Quality",\n)\n\n#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. \nquality_r.to_netcdf(f"../data/NTL_Data_A4/{run_date}_Quality_SL_Berbera_VNP46A4_2012_24.nc")'

# Downloading Monthly Radiance Data

In [ ]:
start_year = 2012
end_year = 2017

# Dictionary to store results
months = []

# Loop through each year
for year in range(start_year, end_year + 1):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 1)

    for i in range(12):
        months.append((start_date + relativedelta(months=i)).strftime("%Y-%m-%d"))

SL_VNP46A3 = bm_raster(
    somaliland_shp_berbera, product_id="VNP46A3", date_range=months, 
    token=bearer
)

filename = f"../data/NTL_Data/{run_date}_SL_Berbera_VNP46A3_EM_{start_year}_{end_year}.nc"
#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
SL_VNP46A3.to_netcdf(filename)

'SL_VNP46A3 = bm_raster(\n    gdf_expanded, product_id="VNP46A3", date_range=months, \n    token=bearer\n)\n\nfilename = f"../data/NTL_Data_A4/{run_date}_SL_expanded_VNP46A3_EM_{start_year}_{end_year}.nc"\nSL_VNP46A3.to_netcdf(filename)'

# Downloading Monthly Quality Data

In [ ]:
Qual_SL_VNP46A3 = bm_raster(
    somaliland_shp_berbera, product_id="VNP46A3", date_range=months, 
    token=bearer,
    variable="NearNadir_Composite_Snow_Free_Quality"
)
filename = f"../data/NTL_Data/{run_date}_Qual_SL_Berbera_VNP46A3_EM_{start_year}_{end_year}.nc"
Qual_SL_VNP46A3.to_netcdf(filename)

'Qual_SL_VNP46A3 = bm_raster(\n    somaliland_shp, product_id="VNP46A3", date_range=months, \n    token=bearer,\n    variable="NearNadir_Composite_Snow_Free_Quality"\n)\nfilename = f"../data/NTL_Data_A4/{run_date}_Qual_SL_VNP46A3_EM_{start_year}_{end_year}.nc"\nQual_SL_VNP46A3.to_netcdf(filename)'

# Downloading Monthly Num Obvs Data

In [ ]:
Num_SL_VNP46A3 = bm_raster(
    somaliland_shp_berbera, product_id="VNP46A3", date_range=months, 
    token=bearer,
    variable="NearNadir_Composite_Snow_Free_Num"
)
filename = f"../data/NTL_Data/{run_date}_Num_SL_Berbera_VNP46A3_EM_{start_year}_{end_year}.nc"

#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
Num_SL_VNP46A3.to_netcdf(filename)


'Num_SL_VNP46A3 = bm_raster(\n    somaliland_shp, product_id="VNP46A3", date_range=months, \n    token=bearer,\n    variable="NearNadir_Composite_Snow_Free_Num"\n)\nfilename = f"../data/NTL_Data_A4/{run_date}_Num_SL_VNP46A3_EM_{start_year}_{end_year}.nc"\nNum_SL_VNP46A3.to_netcdf(filename)'

# Downloading Daily Radiance Data

In [ ]:
# Very Big Data Download
start_year = 2012
end_year = 2015
samples_per_year = 4

# Dictionary to store results
random_days_4_per_year = []

# Loop through each year
for year in range(start_year, end_year + 1):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    delta_days = (end_date - start_date).days

    # Generate unique random days
    random_days = random.sample(range(delta_days + 1), samples_per_year)
    random_dates = [
        (start_date + timedelta(days=day)).strftime("%Y-%m-%d")
        for day in sorted(random_days)]

    random_days_4_per_year.extend(random_dates)

days = bm_raster(
    somaliland_shp_berbera, 
    product_id="VNP46A1", 
    date_range=random_days_4_per_year, 
    token = bearer)

#Check if file saved correctly. Sometimes errors arise after downloading large files, which stops the final line of code. 
days.to_netcdf(f"../data/NTL_Data/{run_date}_random_day_overpass_check_2012_2015.nc")


'start_year = 2012\nend_year = 2015\nsamples_per_year = 4\n\n# Dictionary to store results\nrandom_days_4_per_year = []\n\n# Loop through each year\nfor year in range(start_year, end_year + 1):\n    start_date = datetime(year, 1, 1)\n    end_date = datetime(year, 12, 31)\n    delta_days = (end_date - start_date).days\n\n    # Generate unique random days\n    random_days = random.sample(range(delta_days + 1), samples_per_year)\n    random_dates = [\n        (start_date + timedelta(days=day)).strftime("%Y-%m-%d")\n        for day in sorted(random_days)]\n\n    random_days_4_per_year.extend(random_dates)\n\ndays = bm_raster(\n    somaliland_shp_berbera, \n    product_id="VNP46A1", \n    date_range=random_days_4_per_year, \n    token = bearer)\n\ndays.to_netcdf(f"../data/NTL_Data_A4/{run_date}_random_day_overpass_check_2012_2015.nc")\n'